# Local/Cloud Training for Drowsiness Detection (MobileNetV2)
This notebook is configured to use your custom dataset located at `D:\PROJECT\Drowsiness_detection\final_dataset`.
It trains a single, highly robust **MobileNetV2** model on both Eye and Mouth data simultaneously.

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Must match your prepare_dataset_v2.py
IMG_SIZE = 96
BATCH_SIZE = 32
EPOCHS = 20

# Note: If you run this in Colab, change these paths to your mounted Google Drive paths!
TRAIN_DIR = r'D:\PROJECT\Drowsiness_detection\final_dataset\train'
VAL_DIR = r'D:\PROJECT\Drowsiness_detection\final_dataset\val'


In [ ]:
def build_mobilenet_model():
    # Use MobileNetV2 for highly accurate, lightweight real-time inference
    base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    
    # Fine-tuning: Unfreeze the last few layers of MobileNetV2 for better adaptation
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    # MobileNetV2 expects inputs in [-1, 1]. Keras image_dataset loads as [0, 255].
    # This layer maps [0, 255] to [-1, 1] internally! This makes the exported model foolproof.
    x = tf.keras.layers.Rescaling(scale=1./127.5, offset=-1.0)(inputs)
    
    x = base_model(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs, outputs)
    
    # Use a smaller learning rate since we are fine-tuning
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:
print('Loading Training Data...')
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

print('\nLoading Validation Data...')
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

class_names = train_ds.class_names
print(f'\nDetected Classes: {class_names}')
# Ensure alphabetical order: 0 = closed (drowsy/yawn), 1 = open (awake)

model = build_mobilenet_model()
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('drowsiness_model.keras', monitor='val_accuracy', save_best_only=True)

print('\n🚀 Starting Training...')
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stop, checkpoint])
print('\n✅ Saved drowsiness_model.keras')
